In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data
from sklearn.metrics import classification_report
from torch_geometric.nn import SAGEConv

# ==========================================
# 1. LOAD AND CLEAN DATA
# ==========================================
df = pd.read_csv("wustl-ehms-2020_with_attacks_categories.csv")

drop_cols = ['Dir', 'SrcAddr', 'DstAddr', 'SrcMac', 'DstMac', 'Sport', 'Dport', 'Label', 'Packet_num']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

if 'Flgs' in df.columns:
    df['Flgs'] = LabelEncoder().fit_transform(df['Flgs'].astype(str))

target_enc = LabelEncoder()
y = target_enc.fit_transform(df['Attack Category'])

X_df = df.drop(columns=['Attack Category'], errors='ignore')
X_df = X_df.select_dtypes(include=[np.number]).fillna(0)
X = X_df.values

# ==========================================
# 2. SPLITS AND WEIGHTS
# ==========================================
indices = np.arange(len(df))
# 70% Train, 15% Val, 15% Test
idx_train, idx_temp, y_train, y_temp = train_test_split(indices, y, test_size=0.3, stratify=y, random_state=42)
idx_val, idx_test, y_val, y_test = train_test_split(idx_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)

# ==========================================
# 3. SCALING (Avoid Leakage)
# ==========================================
scaler = StandardScaler()
scaler.fit(X[idx_train])
X_scaled = scaler.transform(X)

# ==========================================
# 4. RANDOM TEST SAMPLE SELECTION
# ==========================================
normal_id = list(target_enc.classes_).index('normal')
attack_ids_in_test = np.where(y_test != normal_id)[0]
random_test_idx = np.random.choice(attack_ids_in_test)
global_idx = idx_test[random_test_idx]

# ==========================================
# 5. GRAPH CONSTRUCTION (Nodes & Edges)
# ==========================================
print("Computing Graph Edges...")
num_nodes = len(X_scaled)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[idx_train] = True
val_mask[idx_val] = True
test_mask[idx_test] = True

# Temporal Edges
source_nodes = np.arange(num_nodes - 1)
target_nodes = np.arange(1, num_nodes)
temporal_edge_index = np.concatenate([np.vstack((source_nodes, target_nodes)), np.vstack((target_nodes, source_nodes))], axis=1)

# KNN Edges
k = 5
knn = NearestNeighbors(n_neighbors=k+1, metric='cosine', n_jobs=-1)
knn.fit(X_scaled)
distances, knn_indices = knn.kneighbors(X_scaled)

knn_sources = np.repeat(np.arange(num_nodes), k)
knn_targets = knn_indices[:, 1:].flatten()
knn_edge_index = np.concatenate([np.vstack((knn_sources, knn_targets)), np.vstack((knn_targets, knn_sources))], axis=1)

# Combine into PyG Data
combined_edges = np.concatenate([temporal_edge_index, knn_edge_index], axis=1)
edge_index_unique = np.unique(combined_edges, axis=1)

data = Data(
    x=torch.tensor(X_scaled, dtype=torch.float),
    edge_index=torch.tensor(edge_index_unique, dtype=torch.long),
    y=torch.tensor(y, dtype=torch.long)
)
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"Graph Data Ready! Nodes: {data.num_nodes}, Edges: {data.num_edges}")

Computing Graph Edges...
Graph Data Ready! Nodes: 16318, Edges: 128208


In [ ]:
class Hybrid_CNNLSTM_GraphSAGE(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Hybrid_CNNLSTM_GraphSAGE, self).__init__()

        # --- CNN Branch ---
        self.conv = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=3, padding=1)
        self.cnn_attn_dense = nn.Linear(128, 1)
        self.cnn_proj = nn.Linear(128, 42)

        # --- LSTM Branch ---
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, batch_first=True)
        self.lstm_attn_dense = nn.Linear(64, 1)
        self.lstm_proj = nn.Linear(64, 42)

        # --- Fusion ---
        self.batch_norm = nn.BatchNorm1d(126)
        self.dropout_cnn = nn.Dropout(p=0.1105)

        # --- GraphSAGE Branch ---
        self.sage1 = SAGEConv(126, 64, aggr='max')
        self.sage2 = SAGEConv(64, 32, aggr='max')
        self.sage3 = SAGEConv(32, num_classes, aggr='max')
        self.dropout_sage = nn.Dropout(p=0.1858)

    def forward(self, x_seq, edge_index):
        # 1. CNN Extraction
        x_cnn = x_seq.transpose(1, 2)
        cnn_b = F.relu(self.conv(x_cnn)).transpose(1, 2)
        cnn_a = F.softmax(torch.tanh(self.cnn_attn_dense(cnn_b)), dim=1)
        cnn_out = F.relu(self.cnn_proj(torch.mean(cnn_b * cnn_a, dim=1)))

        # 2. LSTM Extraction
        lstm_b, _ = self.lstm(x_seq)
        lstm_a = F.softmax(torch.tanh(self.lstm_attn_dense(lstm_b)), dim=1)
        lstm_out = F.relu(self.lstm_proj(torch.mean(lstm_b * lstm_a, dim=1)))

        # 3. Fusion -> Node Embeddings
        fused = cnn_out * lstm_out
        node_embeddings = torch.cat([cnn_out, lstm_out, fused], dim=1)
        node_embeddings = self.dropout_cnn(self.batch_norm(node_embeddings))

        # 4. GraphSAGE Message Passing
        x_graph = self.sage1(node_embeddings, edge_index)
        x_graph = F.relu(x_graph)
        x_graph = self.dropout_sage(x_graph)

        x_graph = self.sage2(x_graph, edge_index)
        x_graph = F.relu(x_graph)
        x_graph = self.dropout_sage(x_graph)

        out = self.sage3(x_graph, edge_index)
        return out

print("Hybrid Architecture Defined!")

Hybrid Architecture Defined!


In [ ]:
# 1. Prepare 3D Data for the CNN+LSTM input (Adds the '1' channel)
data.x_3d = data.x.unsqueeze(-1)

# 2. Initialize Model & Optimizer
hybrid_model = Hybrid_CNNLSTM_GraphSAGE(num_features=data.x.shape[1], num_classes=len(target_enc.classes_))
optimizer = optim.Adam(hybrid_model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# 3. Training and Evaluation Functions
def train_hybrid():
    hybrid_model.train()
    optimizer.zero_grad()
    out = hybrid_model(data.x_3d, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate_hybrid(mask):
    hybrid_model.eval()
    with torch.no_grad():
        out = hybrid_model(data.x_3d, data.edge_index)
        pred = out.argmax(dim=1)
        correct = (pred[mask] == data.y[mask]).sum().item()
        acc = correct / mask.sum().item()
    return acc

# 4. Training Loop
epochs = 300
patience = 20
best_val_loss = float('inf')
patience_counter = 0
best_hybrid_weights = None

print("🔥 Training the End-to-End Hybrid Model 🔥")
for epoch in range(1, epochs + 1):
    loss = train_hybrid()
    val_acc = evaluate_hybrid(data.val_mask)

    hybrid_model.eval()
    with torch.no_grad():
        val_out = hybrid_model(data.x_3d, data.edge_index)
        val_loss = criterion(val_out[data.val_mask], data.y[data.val_mask]).item()

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d} | Train Loss: {loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_hybrid_weights = hybrid_model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}! Restoring best weights.")
        hybrid_model.load_state_dict(best_hybrid_weights)
        break

if best_hybrid_weights is not None:
    hybrid_model.load_state_dict(best_hybrid_weights)

# ==========================================
# 5. FINAL EVALUATION & TEST SAMPLE
# ==========================================
test_acc = evaluate_hybrid(data.test_mask)
print(f'\n🏆 ULTIMATE HYBRID TEST ACCURACY: {test_acc * 100:.2f}% 🏆\n')

hybrid_model.eval()
with torch.no_grad():
    out = hybrid_model(data.x_3d, data.edge_index)
    test_logits = out[data.test_mask]
    test_preds = test_logits.argmax(dim=1).numpy()

test_true = data.y[data.test_mask].numpy()
target_names = [str(c) for c in target_enc.classes_]

print("Classification Report on Ultimate Hybrid Set:")
print(classification_report(test_true, test_preds, target_names=target_names))

# Check Specific Example
target_local_idx = np.where(idx_test == global_idx)[0][0]
sample_logits = test_logits[target_local_idx]
sample_probs = F.softmax(sample_logits, dim=0).numpy()
hybrid_pred = np.argmax(sample_probs)
true_label_id = test_true[target_local_idx]

print("\n" + "="*50)
print(f" HYBRID GRAPH BREAKDOWN FOR TEST SAMPLE {global_idx}")
print("="*50)
print(f"Hybrid Probabilities  : {np.round(sample_probs, 4)}")
print("-" * 50)
print(f"Final Prediction      : {target_enc.classes_[hybrid_pred].upper()}")
print(f"Truth                 : {target_enc.classes_[true_label_id].upper()}")
print("="*50)

🔥 Training the End-to-End Hybrid Model 🔥
Epoch: 010 | Train Loss: 0.5090 | Val Loss: 1.4537 | Val Acc: 0.0690
Epoch: 020 | Train Loss: 0.4684 | Val Loss: 2.4172 | Val Acc: 0.8746

Early stopping at epoch 21! Restoring best weights.

🏆 ULTIMATE HYBRID TEST ACCURACY: 87.46% 🏆

Classification Report on Ultimate Hybrid Set:
                 precision    recall  f1-score   support

Data Alteration       0.00      0.00      0.00       139
       Spoofing       0.00      0.00      0.00       168
         normal       0.87      1.00      0.93      2141

       accuracy                           0.87      2448
      macro avg       0.29      0.33      0.31      2448
   weighted avg       0.76      0.87      0.82      2448


 HYBRID GRAPH BREAKDOWN FOR TEST SAMPLE 1012
Hybrid Probabilities  : [0.0021 0.4573 0.5406]
--------------------------------------------------
Final Prediction      : NORMAL
Truth                 : NORMAL


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
